# Manual Feature Cleaning Review

This notebook reads `merged_feature_matrix.csv`, inspects one energy instrument step by step, and then applies the same cleaning rules to all four energy instruments: `cl1s`, `ho1s`, `rb1s`, and `ng1s`.

The final clean datasets are written to `data/features/clean_feature_set/`. These files are intended as the ready feature inputs for the later modelling/CPCV phase. The notebook does not standardise features, run CPCV, train models, or perform supervised feature selection.

## 1. Imports and paths

`ENERGY_INSTRUMENTS` controls the four datasets created at the end. `INSTRUMENT` is only the instrument shown in the review tables, so the notebook remains readable while the save step still loops through all energy tickers.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 160)

ENERGY_INSTRUMENTS = ["cl1s", "ho1s", "rb1s", "ng1s"]
INSTRUMENT = ENERGY_INSTRUMENTS[0]

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "data" / "features" / "merged_feature_matrix.csv").exists():
        ROOT = candidate
        break

FEATURE_MATRIX_PATH = ROOT / "data" / "features" / "merged_feature_matrix.csv"
FEATURE_CATALOG_PATH = ROOT / "data" / "features" / "feature_catalog.csv"
CLEAN_FEATURE_SET_DIR = ROOT / "data" / "features" / "clean_feature_set"
CLEAN_FEATURE_SET_PATHS = {
    instrument: CLEAN_FEATURE_SET_DIR / f"{instrument}_clean_feature_set.csv"
    for instrument in ENERGY_INSTRUMENTS
}
CLEAN_FEATURE_SET_PATH = CLEAN_FEATURE_SET_PATHS[INSTRUMENT]

HIGH_CORR_THRESHOLD = 0.95

print(f"Review instrument: {INSTRUMENT}")
print(f"Energy instruments: {ENERGY_INSTRUMENTS}")
print(f"Input:             {FEATURE_MATRIX_PATH}")
print(f"Feature catalog:   {FEATURE_CATALOG_PATH}")
print(f"Output folder:     {CLEAN_FEATURE_SET_DIR}")

Review instrument: cl1s
Energy instruments: ['cl1s', 'ho1s', 'rb1s', 'ng1s']
Input:             /Users/faisal/Desktop/sts/data/features/merged_feature_matrix.csv
Feature catalog:   /Users/faisal/Desktop/sts/data/features/feature_catalog.csv
Output folder:     /Users/faisal/Desktop/sts/data/features/clean_feature_set


**After this cell:** Paths are configured. The notebook will show detailed review tables for `INSTRUMENT`, then save one clean dataset per ticker in `ENERGY_INSTRUMENTS`.

## 2. Load data and filter to the review instrument

`merged_feature_matrix.csv` contains all instruments. We check the four energy instruments are present, show their coverage, and then filter to `INSTRUMENT` for the detailed review cells.

In [2]:
full_matrix = pd.read_csv(FEATURE_MATRIX_PATH, parse_dates=["date"])
feature_catalog = pd.read_csv(FEATURE_CATALOG_PATH)
feature_group_map = feature_catalog.set_index("feature_name")["feature_group"].to_dict()

available_instruments = sorted(full_matrix["instrument"].unique())
missing_energy_instruments = sorted(set(ENERGY_INSTRUMENTS) - set(available_instruments))
assert not missing_energy_instruments, f"Missing energy instruments: {missing_energy_instruments}. Available: {available_instruments}"
assert INSTRUMENT in available_instruments, f"{INSTRUMENT} not found. Available: {available_instruments}"

energy_matrix = full_matrix.loc[full_matrix["instrument"].isin(ENERGY_INSTRUMENTS)].copy()
energy_coverage = (
    energy_matrix.groupby("instrument")
    .agg(
        rows=("date", "size"),
        columns=("instrument", lambda s: energy_matrix.shape[1]),
        min_date=("date", "min"),
        max_date=("date", "max"),
        active_signals=("primary_signal", lambda s: int(s.ne(0).sum())),
        duplicate_dates=("date", lambda s: int(s.duplicated().sum())),
    )
    .reindex(ENERGY_INSTRUMENTS)
)

raw = full_matrix.loc[full_matrix["instrument"].eq(INSTRUMENT)].reset_index(drop=True)
duplicate_dates = raw["date"].duplicated().sum()

print("Table: Energy instrument coverage")
display(energy_coverage)

print(f"Table: {INSTRUMENT} dataset summary")
summary = pd.DataFrame(
    {
        "value": [
            len(raw),
            raw.shape[1],
            raw["date"].min(),
            raw["date"].max(),
            duplicate_dates,
        ]
    },
    index=["rows", "columns", "min_date", "max_date", "duplicate_dates"],
)
display(summary)

print(f"Table: {INSTRUMENT} primary_signal distribution")
display(raw["primary_signal"].value_counts(dropna=False).sort_index().to_frame("count"))

print("Table: First rows")
display(raw.head())

Table: Energy instrument coverage


,rows,columns,min_date,max_date,active_signals,duplicate_dates
instrument,,,,,,
cl1s,645,190,2020-01-03,2022-06-30,422,0
ho1s,645,190,2020-01-03,2022-06-30,63,0
rb1s,645,190,2020-01-03,2022-06-30,628,0
ng1s,645,190,2020-01-03,2022-06-30,124,0


Table: cl1s dataset summary


,value
rows,645
columns,190
min_date,2020-01-03 00:00:00
max_date,2022-06-30 00:00:00
duplicate_dates,0


Table: cl1s primary_signal distribution


,count
primary_signal,
-1,36
0,223
1,386


Table: First rows


,date,instrument,primary_signal,open,high,low,close,volume,open_interest,log_ret,ret_1d,ret_2d,ret_3d,ret_5d,ret_10d,ret_20d,ret_63d,ret_126d,ret_252d,mom_sign_5d,mom_sign_20d,mom_sign_63d,mom_sign_126d,mom_sign_252d,roc_5d,roc_20d,roc_63d,ret_spread_5_20,ret_spread_20_63,price_vs_sma5,price_vs_sma10,price_vs_sma20,price_vs_sma50,price_vs_sma100,price_vs_sma200,price_vs_ema5,price_vs_ema10,price_vs_ema20,price_vs_ema50,sma5_vs_sma20,sma10_vs_sma50,sma20_vs_sma50,sma50_vs_sma100,sma50_vs_sma200,sma100_vs_sma200,ema5_vs_ema20,ema12_vs_ema26,ema20_vs_ema50,ema20_vs_ema100,sma20_slope,sma50_slope,sma100_slope,ema20_slope,ema50_slope,macd_line,macd_signal,macd_hist,macd_hist_chg,macd_zscore,vol_5d,vol_10d,vol_20d,vol_63d,vol_126d,vol_252d,ewma_vol_5d,ewma_vol_10d,ewma_vol_20d,ewma_vol_63d,vol_ratio_5_20d,vol_ratio_20_63d,vol_ratio_63_126d,park_vol_5d,park_vol_10d,park_vol_20d,park_vol_63d,gk_vol_5d,gk_vol_10d,gk_vol_20d,gk_vol_63d,hl_range,co_range,intraday_ret,overnight_ret,true_range,atr_5d,atr_5d_pct,atr_10d,atr_10d_pct,atr_14d,...,stoch_d_14d,williams_r_14d,cci_14d,cci_20d,mfi_14d,uo,zscore_20d,zscore_63d,zscore_126d,bb_upper_dist,bb_lower_dist,bb_position,bb_width,bb_width_zscore,donchian_pos_20d,donchian_pos_52d,donchian_pos_252d,hmm_predicted_regime,hmm_regime_confidence,hmm_prob_low_vol,hmm_prob_normal_vol,hmm_prob_high_vol,hmm_prob_extreme_vol,hmm_prob_downside,hmm_prob_weak,hmm_prob_positive,hmm_prob_strong_upside,hmm_prob_stress,hmm_prob_upside_breakout,hmm_prob_calm_positive,hmm_prob_calm_negative,hmm_prob_chop,hmm_prob_high_or_extreme_vol,hmm_prob_low_or_normal_vol,hmm_prob_downside_or_weak,hmm_prob_positive_or_strong_upside,hmm_prob_not_stress,hmm_regime_prob_0,hmm_regime_prob_1,hmm_regime_prob_2,hmm_regime_prob_3,volume_log_change,open_interest_log_change,volume_zscore_20d,volume_zscore_63d,open_interest_zscore_20d,open_interest_zscore_63d,volume_to_open_interest,range_pct,body_pct,upper_wick_pct,lower_wick_pct,close_position_in_bar,gap_open_pct,vol_20d_for_interaction,signal_changed,days_since_signal_change,signal_rolling_mean_5d,signal_rolling_mean_20d,signal_abs_rolling_sum_20d,hmm_regime_missing,hmm_regime_0,hmm_regime_1,hmm_regime_2,hmm_regime_3,hmm_volatility_regime_code,hmm_return_regime_code,hmm_market_state_code,hmm_volatility_regime_missing,hmm_volatility_low_vol,hmm_volatility_normal_vol,hmm_volatility_high_vol,hmm_volatility_extreme_vol,hmm_return_regime_missing,hmm_return_downside,hmm_return_weak,hmm_return_positive,hmm_return_strong_upside,hmm_market_state_missing,hmm_market_stress,hmm_market_upside_breakout,hmm_market_calm_positive,hmm_market_calm_negative,hmm_market_chop,signal_x_hmm_confidence,ret_5d_x_hmm_confidence,vol_20d_x_hmm_confidence,signal_x_hmm_prob_stress,signal_x_hmm_prob_high_or_extreme_vol,signal_x_hmm_prob_positive_or_strong_upside
0,2020-01-03,cl1s,0,24.795579,25.974970,24.775315,25.553469,2.185752e+06,958523.501762,0.030108,0.030566,0.032591,0.022211,0.022211,0.036154,0.080688,0.199550,0.095344,0.299427,1.0,1.0,1.0,1.0,1.0,0.022211,0.080688,0.199550,-0.058477,-0.118861,0.021251,0.027509,0.043218,0.083466,0.110436,0.086984,0.019371,0.027635,0.042866,0.073641,0.021510,0.054459,0.038581,0.024892,0.003247,-0.021120,0.023048,0.019468,0.029510,0.046538,0.003910,0.002946,0.001444,0.004533,0.003015,0.473381,0.443450,0.029931,0.032402,1.310175,0.240319,0.186821,0.152724,0.246618,0.371613,0.337063,0.272244,0.223764,0.219735,0.285221,1.573551,0.619272,0.663642,0.250420,0.199360,0.193527,0.258383,0.257648,0.205057,0.205503,0.263260,1.199655,0.757890,0.030108,0.000000,1.199655,0.561116,0.021959,0.511708,0.020025,0.519856,...,72.541698,-21.186434,205.675985,158.387285,69.962898,60.871319,2.108372,2.135498,2.536731,-0.002129,0.080726,1.027093,0.081993,-0.967451,0.839418,0.907283,0.781953,3.0,0.991140,7.946343e-04,7.739873e-03,9.911404e-01,0.000325,0.000325,7.946343e-04,7.739873e-03,9.911404e-01,0.000325,9.911404e-01,7.739873e-03,7.946343e-04,0.0,0.991465,8.534508e-03,0.001120,9.988803e-01,9.996749e

**After this cell:**

Confirms the dataset for the selected instrument: row count, date range, signal distribution, and no duplicate dates.

## 3. Define ID, signal, and feature columns

`date` and `instrument` are identifiers. `primary_signal` is the trading signal â€” not a feature to clean. Everything else is a feature.

In [3]:
id_cols = ["date", "instrument"]
signal_col = "primary_signal"
feature_cols = [col for col in raw.columns if col not in id_cols + [signal_col]]

numeric_feature_cols = raw[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
non_numeric_feature_cols = [col for col in feature_cols if col not in numeric_feature_cols]

feature_universe = pd.DataFrame(
    {
        "count": [len(id_cols), 1, len(feature_cols), len(numeric_feature_cols), len(non_numeric_feature_cols)]
    },
    index=["id_columns", "signal_columns", "feature_columns", "numeric_feature_columns", "non_numeric_feature_columns"],
)

print("Table: Feature universe summary")
display(feature_universe)

print("Table: Feature count by catalog group")
display(
    pd.Series(feature_cols, name="feature")
    .to_frame()
    .assign(feature_group=lambda x: x["feature"].map(feature_group_map).fillna("uncatalogued"))
    .groupby("feature_group")
    .size()
    .to_frame("feature_count")
    .sort_values("feature_count", ascending=False)
)

print("Non-numeric feature columns:")
print(non_numeric_feature_cols)

Table: Feature universe summary


,count
id_columns,2
signal_columns,1
feature_columns,187
numeric_feature_columns,187
non_numeric_feature_columns,0


Table: Feature count by catalog group


,feature_count
feature_group,
technical,108
engineered_hmm_encoding,21
hmm_category_probability,18
engineered_market,14
engineered_interaction,6
hmm_latent,6
raw_market_data,6
engineered_signal,5
hmm_category_code,3


Non-numeric feature columns:
[]


**After this cell:**

This defines the full feature universe. Non-numeric features stay in the review; they are not silently removed.

## 4. Missingness inspection

Before deciding what to drop, inspect which features are missing and how concentrated the missingness is.

In [4]:
missing_pct = raw[feature_cols].isna().mean()
missing_count = raw[feature_cols].isna().sum()

missingness_table = (
    pd.DataFrame(
        {
            "feature": feature_cols,
            "feature_group": [feature_group_map.get(col, "uncatalogued") for col in feature_cols],
            "missing_count": [int(missing_count[col]) for col in feature_cols],
            "missing_pct": [missing_pct[col] for col in feature_cols],
        }
    )
    .sort_values(["missing_pct", "missing_count", "feature"], ascending=[False, False, True])
    .reset_index(drop=True)
)

print("Table: Features with missing values")
display(missingness_table.loc[missingness_table["missing_count"] > 0])

print("Table: Missingness by feature group")
display(
    missingness_table.groupby("feature_group")
    .agg(
        feature_count=("feature", "size"),
        features_with_missing=("missing_count", lambda s: int((s > 0).sum())),
        max_missing_pct=("missing_pct", "max"),
        mean_missing_pct=("missing_pct", "mean"),
    )
    .sort_values("max_missing_pct", ascending=False)
)

Table: Features with missing values


,feature,feature_group,missing_count,missing_pct
0,hmm_market_calm_negative,engineered_hmm_encoding,19,0.029457
1,hmm_market_calm_positive,engineered_hmm_encoding,19,0.029457
2,hmm_market_chop,engineered_hmm_encoding,19,0.029457
3,hmm_market_state_code,hmm_category_code,19,0.029457
4,hmm_market_stress,engineered_hmm_encoding,19,0.029457
...,...,...,...,...
173,volume_zscore_63d,engineered_market,17,0.026357
174,williams_r_14d,technical,17,0.026357
175,zscore_126d,technical,17,0.026357
176,zscore_20d,technical,17,0.026357


Table: Missingness by feature group


,feature_count,features_with_missing,max_missing_pct,mean_missing_pct
feature_group,,,,
engineered_hmm_encoding,21,17,0.029457,0.023846
engineered_interaction,6,6,0.029457,0.029457
engineered_market,14,14,0.029457,0.026910
hmm_category_code,3,3,0.029457,0.029457
hmm_category_probability,18,18,0.029457,0.029457
hmm_latent,6,6,0.029457,0.029457
raw_market_data,6,6,0.026357,0.026357
technical,108,108,0.026357,0.026357
engineered_signal,5,0,0.000000,0.000000


**After this cell:**

Missingness is tiny in percentage terms, but concentrated. The next section shows the exact rows and context instead of treating missingness as a generic feature-level problem.

## 5. Perfect Duplicate Groups

Exact duplicate features are the easiest first cleaning decision. Choose one feature to keep in each group and mark the rest to drop later. The notebook suggests HMM market-state probability names where those are available, because they are the most interpretable labels.

In [5]:
normalised = raw[feature_cols].astype("string").fillna("<NA>")

parent = {feature: feature for feature in feature_cols}

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    root_a = find(a)
    root_b = find(b)
    if root_a != root_b:
        parent[root_b] = root_a

seen_features = []
duplicate_feature_pairs = []
for col in feature_cols:
    duplicate_of = None
    for previous in seen_features:
        if normalised[col].equals(normalised[previous]):
            duplicate_of = previous
            union(previous, col)
            break
    if duplicate_of is not None:
        duplicate_feature_pairs.append({"feature": col, "duplicate_of": duplicate_of})
    else:
        seen_features.append(col)

duplicate_feature_pairs = pd.DataFrame(duplicate_feature_pairs, columns=["feature", "duplicate_of"])
duplicate_features = set(duplicate_feature_pairs["feature"]) if not duplicate_feature_pairs.empty else set()
duplicate_of_map = dict(zip(duplicate_feature_pairs["feature"], duplicate_feature_pairs["duplicate_of"])) if not duplicate_feature_pairs.empty else {}

clusters = {}
for feature in feature_cols:
    clusters.setdefault(find(feature), []).append(feature)
duplicate_clusters = [sorted(features) for features in clusters.values() if len(features) > 1]

preferred_duplicate_keep_order = [
    "hmm_prob_stress",
    "hmm_prob_calm_positive",
    "hmm_prob_calm_negative",
    "hmm_prob_upside_breakout",
    "hmm_regime_missing",
]

def suggest_duplicate_reason(features: list[str]) -> str:
    joined = " ".join(features)
    if "hmm_prob" in joined or "hmm_regime_prob" in joined:
        return "Exact duplicate HMM probability representation; prefer one interpretable market-state probability name."
    if "hmm_regime_missing" in joined or "_missing" in joined:
        return "Exact duplicate HMM missingness flag; keep one missing flag only."
    if any("hmm_regime_" in f for f in features) or any(f.startswith("hmm_market_") for f in features):
        return "Exact duplicate HMM one-hot/category representation; keep one representation only."
    return "Exact duplicate values across all rows; choose one feature to keep."

def suggested_keep_feature(features: list[str]) -> str:
    for preferred in preferred_duplicate_keep_order:
        if preferred in features:
            return preferred
    market_state_probs = [f for f in features if f.startswith("hmm_prob_") and f in {"hmm_prob_stress", "hmm_prob_calm_positive", "hmm_prob_calm_negative", "hmm_prob_upside_breakout"}]
    if market_state_probs:
        return sorted(market_state_probs)[0]
    return sorted(features)[0]

duplicate_group_rows = []
for group_id, features_in_group in enumerate(duplicate_clusters, start=1):
    suggested_keep = suggested_keep_feature(features_in_group)
    reason = suggest_duplicate_reason(features_in_group)
    for feature in features_in_group:
        duplicate_group_rows.append(
            {
                "duplicate_group_id": group_id,
                "feature": feature,
                "feature_group": feature_group_map.get(feature, "uncatalogued"),
                "missing_pct": missing_pct[feature],
                "unique_values": int(raw[feature].nunique(dropna=True)),
                "suggested_keep": suggested_keep,
                "suggested_reason": reason,
                "manual_keep": "",
                "manual_drop_reason": "",
            }
        )

duplicate_groups_df = pd.DataFrame(duplicate_group_rows)

print(f"Exact duplicate pairs: {len(duplicate_feature_pairs)}")
print(f"Exact duplicate groups: {len(duplicate_clusters)}")
print("Table: Exact duplicate feature pairs")
display(duplicate_feature_pairs)

for group_id in sorted(duplicate_groups_df["duplicate_group_id"].unique()):
    group_table = duplicate_groups_df.loc[duplicate_groups_df["duplicate_group_id"].eq(group_id)].copy()
    print(f"Table: Perfect duplicate group {group_id} | suggested keep = {group_table['suggested_keep'].iloc[0]}")
    display(group_table)

Exact duplicate pairs: 28
Exact duplicate groups: 10
Table: Exact duplicate feature pairs


,feature,duplicate_of
0,hmm_prob_downside,hmm_prob_extreme_vol
1,hmm_prob_weak,hmm_prob_low_vol
2,hmm_prob_positive,hmm_prob_normal_vol
3,hmm_prob_strong_upside,hmm_prob_high_vol
4,hmm_prob_stress,hmm_prob_extreme_vol
5,hmm_prob_upside_breakout,hmm_prob_high_vol
6,hmm_prob_calm_positive,hmm_prob_normal_vol
7,hmm_prob_calm_negative,hmm_prob_low_vol
8,hmm_regime_prob_0,hmm_prob_extreme_vol
9,hmm_regime_prob_1,hmm_prob_normal_vol


Table: Perfect duplicate group 1 | suggested keep = hmm_prob_calm_negative


,duplicate_group_id,feature,feature_group,missing_pct,unique_values,suggested_keep,suggested_reason,manual_keep,manual_drop_reason
0,1,hmm_prob_calm_negative,hmm_category_probability,0.029457,616,hmm_prob_calm_negative,Exact duplicate HMM probability representation...,,
1,1,hmm_prob_low_vol,hmm_category_probability,0.029457,616,hmm_prob_calm_negative,Exact duplicate HMM probability representation...,,
2,1,hmm_prob_weak,hmm_category_probability,0.029457,616,hmm_prob_calm_negative,Exact duplicate HMM probability representation...,,
3,1,hmm_regime_prob_2,hmm_latent,0.029457,616,hmm_prob_calm_negative,Exact duplicate HMM probability representation...,,


Table: Perfect duplicate group 2 | suggested keep = hmm_prob_calm_positive


...

,duplicate_group_id,feature,feature_group,missing_pct,unique_values,suggested_keep,suggested_reason,manual_keep,manual_drop_reason
26,8,hmm_market_calm_positive,engineered_hmm_encoding,0.029457,2,hmm_market_calm_positive,Exact duplicate HMM one-hot/category represent...,,
27,8,hmm_regime_1,engineered_hmm_encoding,0.029457,2,hmm_market_calm_positive,Exact duplicate HMM one-hot/category represent...,,
28,8,hmm_return_positive,engineered_hmm_encoding,0.029457,2,hmm_market_calm_positive,Exact duplicate HMM one-hot/category represent...,,
29,8,hmm_volatility_normal_vol,engineered_hmm_encoding,0.029457,2,hmm_market_calm_positive,Exact duplicate HMM one-hot/category represent...,,


Table: Perfect duplicate group 9 | suggested keep = hmm_market_calm_negative


,duplicate_group_id,feature,feature_group,missing_pct,unique_values,suggested_keep,suggested_reason,manual_keep,manual_drop_reason
30,9,hmm_market_calm_negative,engineered_hmm_encoding,0.029457,2,hmm_market_calm_negative,Exact duplicate HMM one-hot/category represent...,,
31,9,hmm_regime_2,engineered_hmm_encoding,0.029457,2,hmm_market_calm_negative,Exact duplicate HMM one-hot/category represent...,,
32,9,hmm_return_weak,engineered_hmm_encoding,0.029457,2,hmm_market_calm_negative,Exact duplicate HMM one-hot/category represent...,,
33,9,hmm_volatility_low_vol,engineered_hmm_encoding,0.029457,2,hmm_market_calm_negative,Exact duplicate HMM one-hot/category represent...,,


Table: Perfect duplicate group 10 | suggested keep = hmm_market_upside_breakout


,duplicate_group_id,feature,feature_group,missing_pct,unique_values,suggested_keep,suggested_reason,manual_keep,manual_drop_reason
34,10,hmm_market_upside_breakout,engineered_hmm_encoding,0.029457,2,hmm_market_upside_breakout,Exact duplicate HMM one-hot/category represent...,,
35,10,hmm_regime_3,engineered_hmm_encoding,0.029457,2,hmm_market_upside_breakout,Exact duplicate HMM one-hot/category represent...,,
36,10,hmm_return_strong_upside,engineered_hmm_encoding,0.029457,2,hmm_market_upside_breakout,Exact duplicate HMM one-hot/category represent...,,
37,10,hmm_volatility_high_vol,engineered_hmm_encoding,0.029457,2,hmm_market_upside_breakout,Exact duplicate HMM one-hot/category represent...,,


**After this cell:**

This is the first place to make manual decisions. For each duplicate group, pick one feature to keep and record why the others should be dropped later. Nothing is dropped in this notebook.

## 6. Missing Date Investigation

The missing rows are specific calendar events, not broad random missingness. Inspect the dates before deciding whether to drop rows, add a data-quality flag, or impute.

In [6]:
missing_mask = raw[feature_cols].isna()
missing_row_counts = missing_mask.sum(axis=1)
missing_rows_df = raw.loc[missing_row_counts > 0, ["date", "instrument", "primary_signal"]].copy()
missing_rows_df["missing_feature_count"] = missing_row_counts.loc[missing_row_counts > 0].astype(int).to_numpy()
missing_rows_df["missing_feature_names"] = [
    ";".join(missing_mask.columns[row_values].tolist())
    for row_values in missing_mask.loc[missing_row_counts > 0].to_numpy()
]

# Dynamically find the window around the worst missing-value date
if len(missing_rows_df) > 0:
    worst_date = missing_rows_df.sort_values("missing_feature_count", ascending=False)["date"].iloc[0]
    context_start = worst_date - pd.Timedelta(days=4)
    context_end = worst_date + pd.Timedelta(days=4)
    context_df = raw.loc[raw["date"].between(context_start, context_end)].copy()
else:
    context_df = pd.DataFrame()

raw_context_cols = ["date", "instrument", "primary_signal", "open", "high", "low", "close", "volume", "open_interest"]
hmm_context_cols = [
    "date", "instrument", "primary_signal",
    "hmm_predicted_regime", "hmm_regime_confidence",
    "hmm_prob_stress", "hmm_prob_calm_positive", "hmm_prob_calm_negative", "hmm_prob_upside_breakout",
]
engineered_missing_context_cols = [
    "date", "instrument", "volume", "open_interest",
    "volume_log_change", "open_interest_log_change", "volume_to_open_interest",
]

print("Table: Rows with any missing feature values")
display(missing_rows_df)

if not context_df.empty:
    print(f"Table: Raw market context around worst missing date ({worst_date.date()})")
    display(context_df[[col for col in raw_context_cols if col in context_df.columns]])

    print("Table: HMM context around worst missing date")
    display(context_df[[col for col in hmm_context_cols if col in context_df.columns]])

    print("Table: Engineered volume/open-interest context around worst missing date")
    display(context_df[[col for col in engineered_missing_context_cols if col in context_df.columns]])

Table: Rows with any missing feature values


,date,instrument,primary_signal,missing_feature_count,missing_feature_names
11,2020-01-20,cl1s,0,178,open;high;low;close;volume;open_interest;log_r...
31,2020-02-17,cl1s,0,178,open;high;low;close;volume;open_interest;log_r...
100,2020-05-25,cl1s,0,178,open;high;low;close;volume;open_interest;log_r...
129,2020-07-03,cl1s,0,178,open;high;low;close;volume;open_interest;log_r...
175,2020-09-07,cl1s,0,178,open;high;low;close;volume;open_interest;log_r...
182,2020-09-16,cl1s,1,53,hmm_predicted_regime;hmm_regime_confidence;hmm...
183,2020-09-17,cl1s,0,52,hmm_predicted_regime;hmm_regime_confidence;hmm...
233,2020-11-26,cl1s,0,178,open;high;low;close;volume;open_interest;log_r...
268,2021-01-18,cl1s,0,178,open;high;low;close;volume;open_interest;log_r...
288,2021-02-15,cl1s,0,178,open;high;low;close;volume;open_interest;log_r...


Table: Raw market context around worst missing date (2020-01-20)


,date,instrument,primary_signal,open,high,low,close,volume,open_interest
9,2020-01-16,cl1s,0,23.543265,23.851126,23.332624,23.709348,1.052465e+06,1.155954e+06
10,2020-01-17,cl1s,0,23.741755,23.899736,23.620231,23.729602,1.026577e+06,1.132738e+06
11,2020-01-20,cl1s,0,NaN,NaN,NaN,NaN,NaN,NaN
12,2020-01-21,cl1s,0,24.041514,24.211648,23.377182,23.648586,1.600510e+06,1.165979e+06
13,2020-01-22,cl1s,-1,23.599977,23.648586,22.696648,22.984255,1.530855e+06,1.194576e+06
14,2020-01-23,cl1s,-1,22.729054,22.793867,22.186246,22.518412,1.737915e+06,1.183091e+06
15,2020-01-24,cl1s,-1,22.558920,22.664241,21.813573,21.951300,1.447108e+06,1.203433e+06


Table: HMM context around worst missing date


,date,instrument,primary_signal,hmm_predicted_regime,hmm_regime_confidence,hmm_prob_stress,hmm_prob_calm_positive,hmm_prob_calm_negative,hmm_prob_upside_breakout
9,2020-01-16,cl1s,0,2.0,0.991577,0.000183,0.005496,0.991577,0.002744
10,2020-01-17,cl1s,0,2.0,0.999559,0.000103,0.000223,0.999559,0.000115
11,2020-01-20,cl1s,0,NaN,NaN,NaN,NaN,NaN,NaN
12,2020-01-21,cl1s,0,2.0,0.975764,0.015844,0.008032,0.975764,0.000360
13,2020-01-22,cl1s,-1,0.0,0.957739,0.957739,0.007660,0.034521,0.000080
14,2020-01-23,cl1s,-1,2.0,0.928603,0.067596,0.003083,0.928603,0.000718
15,2020-01-24,cl1s,-1,0.0,0.876178,0.876178,0.011067,0.112639,0.000117


Table: Engineered volume/open-interest context around worst missing date


,date,instrument,volume,open_interest,volume_log_change,open_interest_log_change,volume_to_open_interest
9,2020-01-16,cl1s,1.052465e+06,1.155954e+06,0.452372,0.021496,0.910474
10,2020-01-17,cl1s,1.026577e+06,1.132738e+06,-0.024906,-0.020288,0.906279
11,2020-01-20,cl1s,NaN,NaN,NaN,NaN,NaN
12,2020-01-21,cl1s,1.600510e+06,1.165979e+06,0.444093,0.028923,1.372675
13,2020-01-22,cl1s,1.530855e+06,1.194576e+06,-0.044496,0.024230,1.281505
14,2020-01-23,cl1s,1.737915e+06,1.183091e+06,0.126860,-0.009660,1.468961
15,2020-01-24,cl1s,1.447108e+06,1.203433e+06,-0.183119,0.017048,1.202483


**After this cell:**

Missing rows are market holidays (all features NaN) and any HMM-gap dates. The context tables show the surrounding rows so you can confirm the cause before deciding to drop.

## 7. High-Correlation Deep Dive

High correlation is not automatically a drop signal. TA windows are often highly correlated because they describe related horizons. This section separates correlation pairs by type so you can inspect them more sensibly.

In [7]:
numeric_df = raw[numeric_feature_cols]
spearman_corr = numeric_df.corr(method="spearman")
notna_matrix = numeric_df.notna().astype("int8")
pairwise_counts = notna_matrix.T.dot(notna_matrix)

def family_from_feature(feature: str) -> str:
    if feature.startswith("hmm_") or "_hmm_" in feature:
        return "hmm"
    if feature in ["open", "high", "low", "close", "volume", "open_interest"]:
        return "raw_market"
    if any(token in feature for token in ["ret_", "roc_", "mom_", "slope", "sma", "ema", "macd", "rsi", "stoch", "williams", "cci", "mfi", "uo", "zscore", "bb_", "donchian", "atr", "vol_"]):
        return "technical"
    if feature in ["volume_log_change", "open_interest_log_change", "volume_zscore_20d", "volume_zscore_63d", "open_interest_zscore_20d", "open_interest_zscore_63d", "volume_to_open_interest", "range_pct", "body_pct", "upper_wick_pct", "lower_wick_pct", "close_position_in_bar", "gap_open_pct", "vol_20d_for_interaction"]:
        return "engineered_market"
    if feature.startswith("signal_") or feature.startswith("days_since_signal"):
        return "engineered_signal"
    return "other"

def correlation_bucket(row: pd.Series) -> str:
    families = {row["feature_1_family"], row["feature_2_family"]}
    if "hmm" in families:
        return "HMM duplicates/redundancies"
    if families == {"technical"}:
        return "TA window variants"
    if families == {"raw_market"}:
        return "Raw price-level correlations"
    if "technical" in families and "engineered_market" in families:
        return "Engineered-vs-technical overlaps"
    return "Other high-correlation pairs"

high_corr_rows = []
for i, feature_1 in enumerate(numeric_feature_cols):
    for feature_2 in numeric_feature_cols[i + 1:]:
        corr_value = spearman_corr.loc[feature_1, feature_2]
        if pd.isna(corr_value):
            continue
        abs_corr = abs(corr_value)
        if abs_corr >= HIGH_CORR_THRESHOLD:
            high_corr_rows.append(
                {
                    "feature_1": feature_1,
                    "feature_2": feature_2,
                    "feature_1_group": feature_group_map.get(feature_1, "uncatalogued"),
                    "feature_2_group": feature_group_map.get(feature_2, "uncatalogued"),
                    "feature_1_family": family_from_feature(feature_1),
                    "feature_2_family": family_from_feature(feature_2),
                    "spearman_corr": corr_value,
                    "abs_spearman_corr": abs_corr,
                    "n_pairwise_non_missing": int(pairwise_counts.loc[feature_1, feature_2]),
                }
            )

high_corr_pairs = (
    pd.DataFrame(high_corr_rows)
    .sort_values(["abs_spearman_corr", "n_pairwise_non_missing", "feature_1", "feature_2"], ascending=[False, False, True, True])
    .reset_index(drop=True)
    if high_corr_rows
    else pd.DataFrame(columns=["feature_1", "feature_2", "feature_1_group", "feature_2_group", "feature_1_family", "feature_2_family", "spearman_corr", "abs_spearman_corr", "n_pairwise_non_missing"])
)

if not high_corr_pairs.empty:
    high_corr_pairs["correlation_bucket"] = high_corr_pairs.apply(correlation_bucket, axis=1)
    high_corr_pairs["manual_category"] = ""
    high_corr_pairs["manual_action"] = ""
    high_corr_pairs["manual_notes"] = ""
else:
    high_corr_pairs["correlation_bucket"] = []

high_corr_features = set(high_corr_pairs["feature_1"]).union(set(high_corr_pairs["feature_2"])) if not high_corr_pairs.empty else set()
max_abs_spearman_corr = pd.Series(0.0, index=feature_cols)
if not high_corr_pairs.empty:
    stacked = pd.concat(
        [
            high_corr_pairs[["feature_1", "abs_spearman_corr"]].rename(columns={"feature_1": "feature"}),
            high_corr_pairs[["feature_2", "abs_spearman_corr"]].rename(columns={"feature_2": "feature"}),
        ],
        ignore_index=True,
    )
    max_abs_spearman_corr = stacked.groupby("feature")["abs_spearman_corr"].max().reindex(feature_cols).fillna(0.0)

print(f"High-correlation threshold: absolute Spearman >= {HIGH_CORR_THRESHOLD}")
print(f"High-correlation pairs: {len(high_corr_pairs)}")
print(f"Features in at least one high-correlation pair: {len(high_corr_features)}")
if not high_corr_pairs.empty:
    print("Table: High-correlation pair counts by review bucket")
    display(high_corr_pairs.groupby("correlation_bucket").size().to_frame("pair_count").sort_values("pair_count", ascending=False))
    for bucket_name in high_corr_pairs["correlation_bucket"].drop_duplicates().tolist():
        bucket_table = high_corr_pairs.loc[high_corr_pairs["correlation_bucket"].eq(bucket_name)].copy()
        print(f"Table: High-correlation bucket = {bucket_name} | rows = {len(bucket_table)}")
        display(bucket_table)

High-correlation threshold: absolute Spearman >= 0.95
High-correlation pairs: 139
Features in at least one high-correlation pair: 114
Table: High-correlation pair counts by review bucket


,pair_count
correlation_bucket,
HMM duplicates/redundancies,63
TA window variants,58
Other high-correlation pairs,11
Raw price-level correlations,6
Engineered-vs-technical overlaps,1


Table: High-correlation bucket = Other high-correlation pairs | rows = 11


,feature_1,feature_2,feature_1_group,feature_2_group,feature_1_family,feature_2_family,spearman_corr,abs_spearman_corr,n_pairwise_non_missing,correlation_bucket,manual_category,manual_action,manual_notes
0,intraday_ret,body_pct,technical,engineered_market,other,engineered_market,1.000000,1.000000,116,Other high-correlation pairs,,,
1,log_ret,ret_1d,technical,technical,other,technical,1.000000,1.000000,116,Other high-correlation pairs,,,
2,overnight_ret,gap_open_pct,technical,engineered_market,other,engineered_market,1.000000,1.000000,116,Other high-correlation pairs,,,
79,hl_range,true_range,technical,technical,other,other,0.997265,0.997265,116,Other high-correlation pairs,,,
95,co_range,body_pct,technical,engineered_market,other,engineered_market,0.978861,0.978861,116,Other high-correlation pairs,,,
96,co_range,intraday_ret,technical,technical,other,other,0.978861,0.978861,116,Other high-correlation pairs,,,
109,log_ret,rsi_21d_change,technical,technical,other,technical,0.961364,0.961364,116,Other high-correlation pairs,,,
123,log_ret,body_pct,technical,engineered_market,other,engineered_market,0.956539,0.956539,116,Other high-correlation pairs,,,
124,log_ret,intraday_ret,technical,technical,other,other,0.956539,0.956539,116,Other high-correlation pairs,,,
126,ret_1d,intraday_ret,technical,technical,technical,other,0.956539,0.956539,116,Other high-correlation pairs,,,


Table: High-correlation bucket = TA window variants | rows = 58


...

,feature_1,feature_2,feature_1_group,feature_2_group,feature_1_family,feature_2_family,spearman_corr,abs_spearman_corr,n_pairwise_non_missing,correlation_bucket,manual_category,manual_action,manual_notes
10,hmm_prob_calm_negative,hmm_regime_prob_2,hmm_category_probability,hmm_latent,hmm,hmm,1.000000,1.000000,114,HMM duplicates/redundancies,,,
11,hmm_prob_calm_positive,hmm_regime_prob_1,hmm_category_probability,hmm_latent,hmm,hmm,1.000000,1.000000,114,HMM duplicates/redundancies,,,
12,hmm_prob_downside,hmm_prob_stress,hmm_category_probability,hmm_category_probability,hmm,hmm,1.000000,1.000000,114,HMM duplicates/redundancies,,,
13,hmm_prob_downside,hmm_regime_prob_0,hmm_category_probability,hmm_latent,hmm,hmm,1.000000,1.000000,114,HMM duplicates/redundancies,,,
14,hmm_prob_extreme_vol,hmm_prob_downside,hmm_category_probability,hmm_category_probability,hmm,hmm,1.000000,1.000000,114,HMM duplicates/redundancies,,,
15,hmm_prob_extreme_vol,hmm_prob_stress,hmm_category_probability,hmm_category_probability,hmm,hmm,1.000000,1.000000,114,HMM duplicates/redundancies,,,
16,hmm_prob_extreme_vol,hmm_regime_prob_0,hmm_category_probability,hmm_latent,hmm,hmm,1.000000,1.000000,114,HMM duplicates/redundancies,,,
17,hmm_prob_high_vol,hmm_prob_strong_upside,hmm_category_probability,hmm_category_probability,hmm,hmm,1.000000,1.000000,114,HMM duplicates/redundancies,,,
18,hmm_prob_high_vol,hmm_prob_upside_breakout,hmm_category_probability,hmm_category_probability,hmm,hmm,1.000000,1.000000,114,HMM duplicates/redundancies,,,
19,hmm_prob_high_vol,hmm_regime_prob_3,hmm_category_probability,hmm_latent,hmm,hmm,1.000000,1.000000,114,HMM duplicates/redundancies,,,


Table: High-correlation bucket = Raw price-level correlations | rows = 6


,feature_1,feature_2,feature_1_group,feature_2_group,feature_1_family,feature_2_family,spearman_corr,abs_spearman_corr,n_pairwise_non_missing,correlation_bucket,manual_category,manual_action,manual_notes
73,low,close,raw_market_data,raw_market_data,raw_market,raw_market,0.998659,0.998659,116,Raw price-level correlations,,,
75,open,high,raw_market_data,raw_market_data,raw_market,raw_market,0.998389,0.998389,116,Raw price-level correlations,,,
77,high,close,raw_market_data,raw_market_data,raw_market,raw_market,0.997677,0.997677,116,Raw price-level correlations,,,
78,open,low,raw_market_data,raw_market_data,raw_market,raw_market,0.997337,0.997337,116,Raw price-level correlations,,,
80,high,low,raw_market_data,raw_market_data,raw_market,raw_market,0.997214,0.997214,116,Raw price-level correlations,,,
83,open,close,raw_market_data,raw_market_data,raw_market,raw_market,0.996042,0.996042,116,Raw price-level correlations,,,


Table: High-correlation bucket = Engineered-vs-technical overlaps | rows = 1


,feature_1,feature_2,feature_1_group,feature_2_group,feature_1_family,feature_2_family,spearman_corr,abs_spearman_corr,n_pairwise_non_missing,correlation_bucket,manual_category,manual_action,manual_notes
125,ret_1d,body_pct,technical,engineered_market,technical,engineered_market,0.956539,0.956539,116,Engineered-vs-technical overlaps,,,


**After this cell:**

Use the buckets differently. HMM redundancy can often be simplified aggressively. TA window variants need judgement, because nearby windows can be correlated but still represent different horizons.

## 8. Build manual feature review dataframe

`feature_review_df` is the master review surface. It includes automatic issue flags plus blank manual decision columns.

In [8]:
constant_features = set(col for col in feature_cols if raw[col].nunique(dropna=True) <= 1)

def build_issue_flags(row: pd.Series) -> str:
    flags = []
    if row["is_non_numeric"]:
        flags.append("non_numeric")
    if row["missing_pct"] > 0:
        flags.append("missing_values")
    if row["is_constant"]:
        flags.append("constant")
    if row["is_duplicate"]:
        flags.append("duplicate")
    if row["high_corr_flag"]:
        flags.append("high_corr")
    return ";".join(flags)

feature_review_df = pd.DataFrame(
    {
        "feature": feature_cols,
        "feature_group": [feature_group_map.get(col, "uncatalogued") for col in feature_cols],
        "feature_family": [family_from_feature(col) for col in feature_cols],
        "dtype": [str(raw[col].dtype) for col in feature_cols],
        "missing_pct": [missing_pct[col] for col in feature_cols],
        "unique_values": [int(raw[col].nunique(dropna=True)) for col in feature_cols],
        "is_non_numeric": [col in non_numeric_feature_cols for col in feature_cols],
        "is_constant": [col in constant_features for col in feature_cols],
        "is_duplicate": [col in duplicate_features for col in feature_cols],
        "duplicate_of": [duplicate_of_map.get(col, pd.NA) for col in feature_cols],
        "duplicate_group_id": [
            duplicate_groups_df.loc[duplicate_groups_df["feature"].eq(col), "duplicate_group_id"].iloc[0]
            if not duplicate_groups_df.empty and duplicate_groups_df["feature"].eq(col).any()
            else pd.NA
            for col in feature_cols
        ],
        "max_abs_spearman_corr": [float(max_abs_spearman_corr.get(col, 0.0)) for col in feature_cols],
        "high_corr_flag": [col in high_corr_features for col in feature_cols],
    }
)
feature_review_df["auto_issue_flags"] = feature_review_df.apply(build_issue_flags, axis=1)
feature_review_df["manual_category"] = ""
feature_review_df["manual_action"] = ""
feature_review_df["manual_notes"] = ""

feature_review_df = feature_review_df.sort_values(
    ["auto_issue_flags", "feature_group", "feature"],
    ascending=[False, True, True],
).reset_index(drop=True)

print("Table: Manual feature review dataframe")
display(feature_review_df)

print("Table: Auto issue flag counts")
display(
    feature_review_df.assign(has_issue=feature_review_df["auto_issue_flags"].ne(""))
    .groupby(["feature_group", "has_issue"])
    .size()
    .unstack(fill_value=0)
)

Table: Manual feature review dataframe


,feature,feature_group,feature_family,dtype,missing_pct,unique_values,is_non_numeric,is_constant,is_duplicate,duplicate_of,duplicate_group_id,max_abs_spearman_corr,high_corr_flag,auto_issue_flags,manual_category,manual_action,manual_notes
0,hmm_regime_0,engineered_hmm_encoding,hmm,float64,0.029457,2,False,False,False,<NA>,7,1.000000,True,missing_values;high_corr,,,
1,hmm_regime_1,engineered_hmm_encoding,hmm,float64,0.029457,2,False,False,False,<NA>,8,1.000000,True,missing_values;high_corr,,,
2,hmm_regime_2,engineered_hmm_encoding,hmm,float64,0.029457,2,False,False,False,<NA>,9,1.000000,True,missing_values;high_corr,,,
3,hmm_regime_3,engineered_hmm_encoding,hmm,float64,0.029457,2,False,False,False,<NA>,10,1.000000,True,missing_values;high_corr,,,
4,ret_5d_x_hmm_confidence,engineered_interaction,hmm,float64,0.029457,624,False,False,False,<NA>,<NA>,0.996469,True,missing_values;high_corr,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
182,days_since_signal_change,engineered_signal,engineered_signal,int64,0.000000,44,False,False,False,<NA>,<NA>,0.000000,False,,,,
183,signal_abs_rolling_sum_20d,engineered_signal,engineered_signal,float64,0.000000,21,False,False,False,<NA>,<NA>,0.000000,False,,,,
184,signal_changed,engineered_signal,engineered_signal,int64,0.000000,2,False,False,False,<NA>,<NA>,0.000000,False,,,,
185,signal_rolling_mean_20d,engineered_signal,engineered_signal,float64,0.000000,46,False,False,False,<NA>,<NA>,0.000000,False,,,,


Table: Auto issue flag counts


has_issue,False,True
feature_group,,
engineered_hmm_encoding,0,21
engineered_interaction,0,6
engineered_market,0,14
engineered_signal,5,0
hmm_category_code,0,3
hmm_category_probability,0,18
hmm_latent,0,6
raw_market_data,0,6
technical,0,108


**After this cell:**

This is the table to use while categorising features manually. It does not enforce decisions; it only makes the problems visible.

## 9. Feature group review tables

These grouped views make manual inspection easier by showing related feature families together.

In [9]:
for group_name in sorted(feature_review_df["feature_group"].unique()):
    group_table = feature_review_df.loc[feature_review_df["feature_group"].eq(group_name)].copy()
    print(f"Table: Feature review group = {group_name} | rows = {len(group_table)}")
    display(group_table)

Table: Feature review group = engineered_hmm_encoding | rows = 21


,feature,feature_group,feature_family,dtype,missing_pct,unique_values,is_non_numeric,is_constant,is_duplicate,duplicate_of,duplicate_group_id,max_abs_spearman_corr,high_corr_flag,auto_issue_flags,manual_category,manual_action,manual_notes
0,hmm_regime_0,engineered_hmm_encoding,hmm,float64,0.029457,2,False,False,False,<NA>,7,1.0,True,missing_values;high_corr,,,
1,hmm_regime_1,engineered_hmm_encoding,hmm,float64,0.029457,2,False,False,False,<NA>,8,1.0,True,missing_values;high_corr,,,
2,hmm_regime_2,engineered_hmm_encoding,hmm,float64,0.029457,2,False,False,False,<NA>,9,1.0,True,missing_values;high_corr,,,
3,hmm_regime_3,engineered_hmm_encoding,hmm,float64,0.029457,2,False,False,False,<NA>,10,1.0,True,missing_values;high_corr,,,
86,hmm_market_calm_negative,engineered_hmm_encoding,hmm,float64,0.029457,2,False,False,True,hmm_regime_2,9,1.0,True,missing_values;duplicate;high_corr,,,
87,hmm_market_calm_positive,engineered_hmm_encoding,hmm,float64,0.029457,2,False,False,True,hmm_regime_1,8,1.0,True,missing_values;duplicate;high_corr,,,
88,hmm_market_stress,engineered_hmm_encoding,hmm,float64,0.029457,2,False,False,True,hmm_regime_0,7,1.0,True,missing_values;duplicate;high_corr,,,
89,hmm_market_upside_breakout,engineered_hmm_encoding,hmm,float64,0.029457,2,False,False,True,hmm_regime_3,10,1.0,True,missing_values;duplicate;high_corr,,,
90,hmm_return_downside,engineered_hmm_encoding,hmm,float64,0.029457,2,False,False,True,hmm_regime_0,7,1.0,True,missing_values;duplicate;high_corr,,,
91,hmm_return_positive,engineered_hmm_encoding,hmm,float64,0.029457,2,False,False,True,hmm_regime_1,8,1.0,True,missing_values;duplicate;high_corr,,,


Table: Feature review group = engineered_interaction | rows = 6


,feature,feature_group,feature_family,dtype,missing_pct,unique_values,is_non_numeric,is_constant,is_duplicate,duplicate_of,duplicate_group_id,max_abs_spearman_corr,high_corr_flag,auto_issue_flags,manual_category,manual_action,manual_notes
4,ret_5d_x_hmm_confidence,engineered_interaction,hmm,float64,0.029457,624,False,False,False,<NA>,<NA>,0.996469,True,missing_values;high_corr,,,
112,signal_x_hmm_confidence,engineered_interaction,hmm,float64,0.029457,390,False,False,False,<NA>,<NA>,0.000000,False,missing_values,,,
113,signal_x_hmm_prob_high_or_extreme_vol,engineered_interaction,hmm,float64,0.029457,416,False,False,False,<NA>,<NA>,0.000000,False,missing_values,,,
114,signal_x_hmm_prob_positive_or_strong_upside,engineered_interaction,hmm,float64,0.029457,405,False,False,False,<NA>,<NA>,0.000000,False,missing_values,,,
115,signal_x_hmm_prob_stress,engineered_interaction,hmm,float64,0.029457,409,False,False,False,<NA>,<NA>,0.000000,False,missing_values,,,
116,vol_20d_x_hmm_confidence,engineered_interaction,hmm,float64,0.029457,626,False,False,False,<NA>,<NA>,0.000000,False,missing_values,,,


Table: Feature review group = engineered_market | rows = 14


...

,feature,feature_group,feature_family,dtype,missing_pct,unique_values,is_non_numeric,is_constant,is_duplicate,duplicate_of,duplicate_group_id,max_abs_spearman_corr,high_corr_flag,auto_issue_flags,manual_category,manual_action,manual_notes
106,hmm_regime_prob_0,hmm_latent,hmm,float64,0.029457,602,False,False,True,hmm_prob_extreme_vol,4,1.0,True,missing_values;duplicate;high_corr,,,
107,hmm_regime_prob_1,hmm_latent,hmm,float64,0.029457,598,False,False,True,hmm_prob_normal_vol,2,1.0,True,missing_values;duplicate;high_corr,,,
108,hmm_regime_prob_2,hmm_latent,hmm,float64,0.029457,616,False,False,True,hmm_prob_low_vol,1,1.0,True,missing_values;duplicate;high_corr,,,
109,hmm_regime_prob_3,hmm_latent,hmm,float64,0.029457,626,False,False,True,hmm_prob_high_vol,3,1.0,True,missing_values;duplicate;high_corr,,,
129,hmm_predicted_regime,hmm_latent,hmm,float64,0.029457,4,False,False,False,<NA>,<NA>,0.0,False,missing_values,,,
130,hmm_regime_confidence,hmm_latent,hmm,float64,0.029457,573,False,False,False,<NA>,<NA>,0.0,False,missing_values,,,


Table: Feature review group = raw_market_data | rows = 6


,feature,feature_group,feature_family,dtype,missing_pct,unique_values,is_non_numeric,is_constant,is_duplicate,duplicate_of,duplicate_group_id,max_abs_spearman_corr,high_corr_flag,auto_issue_flags,manual_category,manual_action,manual_notes
19,close,raw_market_data,raw_market,float64,0.026357,616,False,False,False,<NA>,<NA>,0.998659,True,missing_values;high_corr,,,
20,high,raw_market_data,raw_market,float64,0.026357,620,False,False,False,<NA>,<NA>,0.998389,True,missing_values;high_corr,,,
21,low,raw_market_data,raw_market,float64,0.026357,625,False,False,False,<NA>,<NA>,0.998659,True,missing_values;high_corr,,,
22,open,raw_market_data,raw_market,float64,0.026357,624,False,False,False,<NA>,<NA>,0.998389,True,missing_values;high_corr,,,
131,open_interest,raw_market_data,raw_market,float64,0.026357,628,False,False,False,<NA>,<NA>,0.000000,False,missing_values,,,
132,volume,raw_market_data,raw_market,float64,0.026357,628,False,False,False,<NA>,<NA>,0.000000,False,missing_values,,,


Table: Feature review group = technical | rows = 108


,feature,feature_group,feature_family,dtype,missing_pct,unique_values,is_non_numeric,is_constant,is_duplicate,duplicate_of,duplicate_group_id,max_abs_spearman_corr,high_corr_flag,auto_issue_flags,manual_category,manual_action,manual_notes
23,atr_10d,technical,technical,float64,0.026357,628,False,False,False,<NA>,<NA>,0.993705,True,missing_values;high_corr,,,
24,atr_10d_pct,technical,technical,float64,0.026357,628,False,False,False,<NA>,<NA>,0.989795,True,missing_values;high_corr,,,
25,atr_14d,technical,technical,float64,0.026357,628,False,False,False,<NA>,<NA>,0.993705,True,missing_values;high_corr,,,
26,atr_14d_pct,technical,technical,float64,0.026357,628,False,False,False,<NA>,<NA>,0.989795,True,missing_values;high_corr,,,
27,atr_20d,technical,technical,float64,0.026357,628,False,False,False,<NA>,<NA>,0.991816,True,missing_values;high_corr,,,
28,atr_20d_pct,technical,technical,float64,0.026357,628,False,False,False,<NA>,<NA>,0.986695,True,missing_values;high_corr,,,
29,atr_5d,technical,technical,float64,0.026357,628,False,False,False,<NA>,<NA>,0.979292,True,missing_values;high_corr,,,
30,atr_5d_pct,technical,technical,float64,0.026357,628,False,False,False,<NA>,<NA>,0.964402,True,missing_values;high_corr,,,
31,bb_position,technical,technical,float64,0.026357,628,False,False,False,<NA>,<NA>,1.000000,True,missing_values;high_corr,,,
32,cci_14d,technical,technical,float64,0.026357,628,False,False,False,<NA>,<NA>,0.955235,True,missing_values;high_corr,,,


**After this cell:**

Reviewing by feature group is less overwhelming than scanning all features at once. Start with HMM groups and duplicate groups before worrying about TA window correlations.

## 10. Save cleaned model datasets

Applies the cleaning decisions that are obvious from the review tables: exact HMM aliases, constant HMM columns, duplicate missing flags, non-numeric labels, and simple TA overlaps. The same column-cleaning rules are applied to each energy instrument in `ENERGY_INSTRUMENTS`. Rows with any remaining missing feature values are dropped per instrument. Output is saved to `data/features/clean_feature_set/{instrument}_clean_feature_set.csv`.

In [10]:
hmm_constant_drop_features = ["hmm_prob_chop", "hmm_market_chop"]

hmm_missing_flag_drop_features = [
    "hmm_volatility_regime_missing",
    "hmm_return_regime_missing",
    "hmm_market_state_missing",
]

hmm_probability_alias_drop_features = [
    "hmm_prob_extreme_vol", "hmm_prob_downside", "hmm_regime_prob_0",
    "hmm_prob_normal_vol", "hmm_prob_positive", "hmm_regime_prob_1",
    "hmm_prob_low_vol", "hmm_prob_weak", "hmm_regime_prob_2",
    "hmm_prob_high_vol", "hmm_prob_strong_upside", "hmm_regime_prob_3",
    "hmm_prob_not_stress",
]

hmm_one_hot_alias_drop_features = [
    "hmm_regime_0", "hmm_regime_1", "hmm_regime_2", "hmm_regime_3",
    "hmm_volatility_extreme_vol", "hmm_volatility_normal_vol",
    "hmm_volatility_low_vol", "hmm_volatility_high_vol",
    "hmm_return_downside", "hmm_return_positive",
    "hmm_return_weak", "hmm_return_strong_upside",
]

non_numeric_hmm_label_drop_features = [
    "hmm_volatility_regime", "hmm_return_regime", "hmm_market_state",
]

hmm_category_code_drop_features = [
    "hmm_volatility_regime_code", "hmm_return_regime_code", "hmm_market_state_code",
]

simple_ta_alias_drop_features = [
    "roc_5d", "roc_20d", "roc_63d", "bb_position", "overnight_ret", "intraday_ret",
]

return_ladder_drop_features = ["log_ret", "ret_2d", "ret_3d", "ret_10d"]

requested_hard_drop_features = (
    hmm_constant_drop_features
    + hmm_missing_flag_drop_features
    + hmm_probability_alias_drop_features
    + hmm_one_hot_alias_drop_features
    + non_numeric_hmm_label_drop_features
    + hmm_category_code_drop_features
    + simple_ta_alias_drop_features
    + return_ladder_drop_features
)
hard_drop_features = [f for f in requested_hard_drop_features if f in full_matrix.columns]

id_cols = ["date", "instrument"]
signal_col = "primary_signal"
CLEAN_FEATURE_SET_DIR.mkdir(parents=True, exist_ok=True)

clean_feature_sets = {}
clean_dataset_summaries = []

for instrument in ENERGY_INSTRUMENTS:
    instrument_raw = full_matrix.loc[full_matrix["instrument"].eq(instrument)].reset_index(drop=True)
    assert len(instrument_raw) > 0, f"No rows found for {instrument}."
    assert instrument_raw["date"].duplicated().sum() == 0, f"Duplicate dates found for {instrument}."

    instrument_clean = instrument_raw.drop(columns=hard_drop_features).copy()
    instrument_model_feature_cols = [col for col in instrument_clean.columns if col not in id_cols + [signal_col]]
    missing_after_drop = instrument_clean[instrument_model_feature_cols].isna().any(axis=1)
    instrument_clean = instrument_clean.loc[~missing_after_drop].reset_index(drop=True)

    output_path = CLEAN_FEATURE_SET_PATHS[instrument]
    instrument_clean.to_csv(output_path, index=False)
    clean_feature_sets[instrument] = instrument_clean

    clean_dataset_summaries.append(
        {
            "instrument": instrument,
            "original_rows": len(instrument_raw),
            "clean_rows": len(instrument_clean),
            "rows_removed": len(instrument_raw) - len(instrument_clean),
            "original_columns": instrument_raw.shape[1],
            "clean_columns": instrument_clean.shape[1],
            "features_removed": len(hard_drop_features),
            "remaining_feature_columns": len(instrument_model_feature_cols),
            "output_path": str(output_path),
        }
    )

# Keep the review instrument variables available for the validation cells below.
clean = clean_feature_sets[INSTRUMENT]
model_feature_cols = [col for col in clean.columns if col not in id_cols + [signal_col]]
CLEAN_FEATURE_SET_PATH = CLEAN_FEATURE_SET_PATHS[INSTRUMENT]

hard_drop_summary = pd.DataFrame(
    {
        "drop_group": (
            ["constant_hmm"] * len(hmm_constant_drop_features)
            + ["duplicate_missing_flag"] * len(hmm_missing_flag_drop_features)
            + ["hmm_probability_alias"] * len(hmm_probability_alias_drop_features)
            + ["hmm_one_hot_alias"] * len(hmm_one_hot_alias_drop_features)
            + ["non_numeric_hmm_label"] * len(non_numeric_hmm_label_drop_features)
            + ["hmm_category_code"] * len(hmm_category_code_drop_features)
            + ["simple_ta_alias"] * len(simple_ta_alias_drop_features)
            + ["return_ladder_trim"] * len(return_ladder_drop_features)
        ),
        "feature": requested_hard_drop_features,
    }
).loc[lambda df: df["feature"].isin(hard_drop_features)].reset_index(drop=True)

clean_dataset_summary = pd.DataFrame(clean_dataset_summaries)
model_dataset_summary = clean_dataset_summary.loc[clean_dataset_summary["instrument"].eq(INSTRUMENT)].T.rename(columns={0: "value"})

print("Table: Hard-drop feature list")
display(hard_drop_summary)

print("Table: Clean dataset summary for all energy instruments")
display(clean_dataset_summary)

print(f"Table: Clean dataset sample for {INSTRUMENT}")
display(clean.head())

print("Saved clean feature sets:")
for instrument, path in CLEAN_FEATURE_SET_PATHS.items():
    print(f"- {instrument}: {path}")

Table: Hard-drop feature list


,drop_group,feature
0,constant_hmm,hmm_prob_chop
1,constant_hmm,hmm_market_chop
2,duplicate_missing_flag,hmm_volatility_regime_missing
3,duplicate_missing_flag,hmm_return_regime_missing
4,duplicate_missing_flag,hmm_market_state_missing
5,hmm_probability_alias,hmm_prob_extreme_vol
6,hmm_probability_alias,hmm_prob_downside
7,hmm_probability_alias,hmm_regime_prob_0
8,hmm_probability_alias,hmm_prob_normal_vol
9,hmm_probability_alias,hmm_prob_positive


Table: Clean dataset summary for all energy instruments


,instrument,original_rows,clean_rows,rows_removed,original_columns,clean_columns,features_removed,remaining_feature_columns,output_path
0,cl1s,645,626,19,190,147,43,144,/Users/faisal/Desktop/sts/data/features/clean_...
1,ho1s,645,628,17,190,147,43,144,/Users/faisal/Desktop/sts/data/features/clean_...
2,rb1s,645,628,17,190,147,43,144,/Users/faisal/Desktop/sts/data/features/clean_...
3,ng1s,645,628,17,190,147,43,144,/Users/faisal/Desktop/sts/data/features/clean_...


Table: Clean dataset sample for cl1s


,date,instrument,primary_signal,open,high,low,close,volume,open_interest,ret_1d,ret_5d,ret_20d,ret_63d,ret_126d,ret_252d,mom_sign_5d,mom_sign_20d,mom_sign_63d,mom_sign_126d,mom_sign_252d,ret_spread_5_20,ret_spread_20_63,price_vs_sma5,price_vs_sma10,price_vs_sma20,price_vs_sma50,price_vs_sma100,price_vs_sma200,price_vs_ema5,price_vs_ema10,price_vs_ema20,price_vs_ema50,sma5_vs_sma20,sma10_vs_sma50,sma20_vs_sma50,sma50_vs_sma100,sma50_vs_sma200,sma100_vs_sma200,ema5_vs_ema20,ema12_vs_ema26,ema20_vs_ema50,ema20_vs_ema100,sma20_slope,sma50_slope,sma100_slope,ema20_slope,ema50_slope,macd_line,macd_signal,macd_hist,macd_hist_chg,macd_zscore,vol_5d,vol_10d,vol_20d,vol_63d,vol_126d,vol_252d,ewma_vol_5d,ewma_vol_10d,ewma_vol_20d,ewma_vol_63d,vol_ratio_5_20d,vol_ratio_20_63d,vol_ratio_63_126d,park_vol_5d,park_vol_10d,park_vol_20d,park_vol_63d,gk_vol_5d,gk_vol_10d,gk_vol_20d,gk_vol_63d,hl_range,co_range,true_range,atr_5d,atr_5d_pct,atr_10d,atr_10d_pct,atr_14d,atr_14d_pct,atr_20d,atr_20d_pct,rsi_7d,rsi_7d_change,rsi_14d,rsi_14d_change,rsi_21d,rsi_21d_change,stoch_k_14d,stoch_d_14d,williams_r_14d,cci_14d,cci_20d,mfi_14d,uo,zscore_20d,zscore_63d,zscore_126d,bb_upper_dist,bb_lower_dist,bb_width,bb_width_zscore,donchian_pos_20d,donchian_pos_52d,donchian_pos_252d,hmm_predicted_regime,hmm_regime_confidence,hmm_prob_stress,hmm_prob_upside_breakout,hmm_prob_calm_positive,hmm_prob_calm_negative,hmm_prob_high_or_extreme_vol,hmm_prob_low_or_normal_vol,hmm_prob_downside_or_weak,hmm_prob_positive_or_strong_upside,volume_log_change,open_interest_log_change,volume_zscore_20d,volume_zscore_63d,open_interest_zscore_20d,open_interest_zscore_63d,volume_to_open_interest,range_pct,body_pct,upper_wick_pct,lower_wick_pct,close_position_in_bar,gap_open_pct,vol_20d_for_interaction,signal_changed,days_since_signal_change,signal_rolling_mean_5d,signal_rolling_mean_20d,signal_abs_rolling_sum_20d,hmm_regime_missing,hmm_market_stress,hmm_market_upside_breakout,hmm_market_calm_positive,hmm_market_calm_negative,signal_x_hmm_confidence,ret_5d_x_hmm_confidence,vol_20d_x_hmm_confidence,signal_x_hmm_prob_stress,signal_x_hmm_prob_high_or_extreme_vol,signal_x_hmm_prob_positive_or_strong_upside
0,2020-01-03,cl1s,0,24.795579,25.974970,24.775315,25.553469,2.185752e+06,958523.501762,0.030566,0.022211,0.080688,0.199550,0.095344,0.299427,1.0,1.0,1.0,1.0,1.0,-0.058477,-0.118861,0.021251,0.027509,0.043218,0.083466,0.110436,0.086984,0.019371,0.027635,0.042866,0.073641,0.021510,0.054459,0.038581,0.024892,0.003247,-0.021120,0.023048,0.019468,0.029510,0.046538,0.003910,0.002946,0.001444,0.004533,0.003015,0.473381,0.443450,0.029931,0.032402,1.310175,0.240319,0.186821,0.152724,0.246618,0.371613,0.337063,0.272244,0.223764,0.219735,0.285221,1.573551,0.619272,0.663642,0.250420,0.199360,0.193527,0.258383,0.257648,0.205057,0.205503,0.263260,1.199655,0.757890,1.199655,0.561116,0.021959,0.511708,0.020025,0.519856,0.020344,0.543275,0.021260,78.102827,18.324875,70.179907,8.628436,65.494265,5.618253,78.813566,72.541698,-21.186434,205.675985,158.387285,69.962898,60.871319,2.108372,2.135498,2.536731,-0.002129,0.080726,0.081993,-0.967451,0.839418,0.907283,0.781953,3.0,0.991140,0.000325,9.911404e-01,7.739873e-03,7.946343e-04,0.991465,8.534508e-03,0.001120,9.988803e-01,0.598557,-0.022820,2.469962,2.781207,0.174642,0.162133,2.280332,0.046947,0.030566,0.016495,0.000793,0.648649,0.000000,0.009692,0,0,0.000000,0.000000,0.0,0,0.0,1.0,0.0,0.0,0.000000,0.022015,0.009606,0.000000,0.000000,0.000000
1,2020-01-06,cl1s,0,25.820960,26.230302,25.387301,25.642633,1.786962e+06,909240.146872,0.003489,0.025113,0.084459,0.195529,0.096306,0.280307,1.0,1.0,1.0,1.0,1.0,-0.059346,-0.111070,0.019694,0.027594,0.042608,0.084535,0.113088,0.090535,0.015170,0.025400,0.041891,0.074127,0.022471,0.055412,0.040214,0.026328,0.005532,-0.020262,0.026322,0.020739,0.030941,0.049195,0.004077,0.002501,0.001099,0.004429,0.003035,0.506348,0.456029,0.050319,0.020387,1.392643,0.238370,0.186535,0.152049,0.246491,0.371625,0.336626,0.228189,0.203104,0

Saved clean feature sets:
- cl1s: /Users/faisal/Desktop/sts/data/features/clean_feature_set/cl1s_clean_feature_set.csv
- ho1s: /Users/faisal/Desktop/sts/data/features/clean_feature_set/ho1s_clean_feature_set.csv
- rb1s: /Users/faisal/Desktop/sts/data/features/clean_feature_set/rb1s_clean_feature_set.csv
- ng1s: /Users/faisal/Desktop/sts/data/features/clean_feature_set/ng1s_clean_feature_set.csv


**After this cell:** Clean datasets are saved in `data/features/clean_feature_set/`. Duplicate/alias features are removed and rows with missing values are dropped separately for each energy instrument. High-correlation TA families that are not simple aliases remain for later CPCV-safe selection.

## 11. Validation

Confirms every energy clean dataset was saved correctly: no duplicate dates, no remaining missing feature values, no hard-dropped columns, and `primary_signal` is retained.

In [11]:
assert INSTRUMENT in full_matrix["instrument"].values, f"{INSTRUMENT} not found in feature matrix."
assert len(raw) > 0, "No rows after filtering to instrument."
assert duplicate_dates == 0, "Duplicate dates found for review instrument."
assert len(feature_review_df) == len(feature_cols), "feature_review_df should have one row per feature column."
assert set(feature_review_df["feature"]) == set(feature_cols), "feature_review_df feature set does not match feature columns."
assert "feature_review_df" in globals(), "feature_review_df was not created."
assert "high_corr_pairs" in globals(), "high_corr_pairs was not created."
assert "duplicate_feature_pairs" in globals(), "duplicate_feature_pairs was not created."
assert "duplicate_groups_df" in globals(), "duplicate_groups_df was not created."
assert "clean_feature_sets" in globals(), "clean_feature_sets was not created."

validation_rows = []
for instrument in ENERGY_INSTRUMENTS:
    path = CLEAN_FEATURE_SET_PATHS[instrument]
    assert path.exists(), f"Output file was not saved: {path}"

    saved = pd.read_csv(path, parse_dates=["date"])
    saved_feature_cols = [col for col in saved.columns if col not in id_cols + [signal_col]]

    assert len(saved) > 0, f"Clean dataset is empty for {instrument}."
    assert saved["instrument"].eq(instrument).all(), f"Unexpected instrument values in {path}."
    assert saved["date"].duplicated().sum() == 0, f"Duplicate dates found in {path}."
    assert not set(hard_drop_features).intersection(saved.columns), f"Hard-drop features still present in {path}."
    assert saved[saved_feature_cols].isna().sum().sum() == 0, f"Clean dataset still has missing feature values: {path}."
    assert signal_col in saved.columns, f"primary_signal column is missing from {path}."

    validation_rows.append(
        {
            "instrument": instrument,
            "rows": len(saved),
            "columns": saved.shape[1],
            "feature_columns": len(saved_feature_cols),
            "min_date": saved["date"].min(),
            "max_date": saved["date"].max(),
            "primary_signal_values": sorted(saved[signal_col].dropna().unique().tolist()),
            "path": str(path),
        }
    )

validation_summary = pd.DataFrame(validation_rows)
print("Validation passed for all clean energy feature sets.")
display(validation_summary)

Validation passed for all clean energy feature sets.


,instrument,rows,columns,feature_columns,min_date,max_date,primary_signal_values,path
0,cl1s,626,147,144,2020-01-03,2022-06-30,"[-1, 0, 1]",/Users/faisal/Desktop/sts/data/features/clean_...
1,ho1s,628,147,144,2020-01-03,2022-06-30,"[-1, 0, 1]",/Users/faisal/Desktop/sts/data/features/clean_...
2,rb1s,628,147,144,2020-01-03,2022-06-30,"[-1, 1]",/Users/faisal/Desktop/sts/data/features/clean_...
3,ng1s,628,147,144,2020-01-03,2022-06-30,"[-1, 0]",/Users/faisal/Desktop/sts/data/features/clean_...


**After this cell:** All four clean feature sets are ready for the modelling phase and available from `clean_feature_sets` in memory or from the CSV files under `data/features/clean_feature_set/`.